In [1]:
import sys
import pickle
sys.path.append('../../TaskExecutionTimeMining/')
from mutual_information import *
from information_bottleneck import *
from sklearn.metrics import mutual_info_score
from sklearn.feature_selection import mutual_info_regression

import pandas as pd
import numpy as np
#np.seterr(divide='ignore', invalid='ignore')

In [2]:
with open("../transformed_event_logs/BPIC_2017_all_train.pickle", "rb") as f:
    event_log = pickle.load(f)

# numerical attributes : duration, seconds_in_day, day_in_week
numerical_attributes = [
    'duration_seconds',
    'seconds_in_day',
    'day_of_week',
    'case:RequestedAmount_start'
]

transformed_event_log = event_log.copy()

for num_attr in numerical_attributes:
    transformed_event_log[num_attr] = np.log(transformed_event_log[num_attr]+1)
    transformed_event_log[num_attr] = (transformed_event_log[num_attr] - transformed_event_log[num_attr].mean()) / transformed_event_log[num_attr].std()

/tmp/ipykernel_381914/2135300804.py:2: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  event_log = pickle.load(f)


In [9]:
transformed_event_log

,Action_start,org:resource_start,concept:name,EventOrigin_start,EventID_start,lifecycle:transition_start,time:timestamp_start,case:LoanGoal_start,case:ApplicationType_start,case:concept:name,...,W_Shortened completion __resume,W_Shortened completion __schedule,W_Shortened completion __start,W_Shortened completion __suspend,W_Validate application__ate_abort,W_Validate application__complete,W_Validate application__resume,W_Validate application__schedule,W_Validate application__start,W_Validate application__suspend
1,Released,User_63,W_Call after offers__suspend,Workflow,Workitem_1000010198,suspend,2016-08-20 12:19:22.434000+00:00,Home improvement,New credit,Application_1930272371,...,0,0,0,0,0,0,0,0,0,0
2,Obtained,User_80,W_Call after offers__start,Workflow,Workitem_1000013868,start,2016-10-18 08:54:52.604000+00:00,Car,New credit,Application_1804686886,...,0,0,0,0,0,0,0,0,0,0
4,Obtained,User_114,W_Validate application__start,Workflow,Workitem_1000015916,start,2016-01-19 12:58:12.261000+00:00,Car,New credit,Application_704665572,...,0,0,0,0,0,0,0,1,1,0
5,Obtained,User_51,W_Call incomplete files__resume,Workflow,Workitem_1000016777,resume,2016-12-01 19:01:26.846000+00:00,Home improvement,New credit,Application_907188218,...,0,0,0,0,0,1,0,1,1,0
6,Obtained,User_5,W_Call after offers__resume,Workflow,Workitem_1000019350,resume,2016-02-13 15:34:25.947000+00:00,Home improvement,New credit,Application_2110398373,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
619777,Released,User_19,W_Call after offers__suspend,Workflow,Workitem_999988201,suspend,2016-10-04 18:43:06.137000+00:00,Home improvement,New credit,Application_1551324804,...,0,0,0,0,0,0,0,0,0,0
619778,Obtained,User_28,W_Call incomplete files__resume,Workflow,Workitem_999990412,resume,2016-04-01 18:08:48.145000+00:00,Car,New credit,Application_883995052,...,0,0,0,0,1,0,0,1,1,1
619779,Released,User_51,W_Call after offers__suspend,Workflow,Workitem_999991648,suspend,2016-12-01 20:02:50.875000+00:00,Not speficied,New credit,Application_1000610355,...,0,0,0,0,0,0,0,0,0,0
619780,Created,User_118,W_Validate application__schedule,Workflow,Workitem_99999173,schedule,2016-04-13 07:49:14.822000+00:00,Not speficied,New credit,Application_52539020,...,0,0,0,0,0,0,0,1,0,0


In [ ]:
for n in [2, 4, 8, 16, 32, 64, 128, 256]:#, 512, 1024]:
    print(f"n_neighbors={n}: {mutual_info_regression(transformed_event_log[['seconds_in_day']], transformed_event_log['duration_seconds'], n_neighbors=n)}")

In [7]:
for n in [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096, 8192, 8192*2, 32768, 65536, 131072, 262144, 524288, 1048576]:
    x_discrete = pd.cut(transformed_event_log['seconds_in_day'], bins=n, labels=False)
    y_discrete = pd.cut(transformed_event_log['duration_seconds'], bins=n, labels=False)

    # Compute MI
    print(f"bins={n}: {mutual_info_score(x_discrete, y_discrete)}")

bins=2: 3.372954906067575e-05
bins=4: 0.0001962677969524626
bins=8: 0.008846262545584758
bins=16: 0.03788353282090677
bins=32: 0.06580181320374269
bins=64: 0.0933262687993967
bins=128: 0.12631330837907997
bins=256: 0.16141824425422174
bins=512: 0.21386818529111498
bins=1024: 0.3502523223692623
bins=2048: 0.7347193196618366
bins=4096: 1.5035627112401357
bins=8192: 2.510740437091683
bins=16384: 3.6265489735284566
bins=32768: 4.760585433891558
bins=65536: 5.891951876404759
bins=131072: 6.985016166145227
bins=262144: 7.948048074812444
bins=524288: 8.449926185925444
bins=1048576: 8.660423785496684


/home/LordKunkler/.local/share/virtualenvs/TaskExecutionTimeMining-yRnjZRF7/lib/python3.11/site-packages/sklearn/metrics/cluster/_supervised.py:50: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  type_pred = type_of_target(labels_pred)


In [ ]:
print("Cluster Assignments:", cluster_assignments)

In [ ]:
print(cluster_ranges)

In [ ]:
transformed_event_log['seconds_in_day'].hist(bins=128)

In [ ]:
for n in [2, 4, 8, 16, 32, 64]:#, 128, 256, 512]:
    print(f"bins={n}: {MI_continuous_continuous(transformed_event_log, 'seconds_in_day', 'duration_seconds', n)}")

In [ ]:
for n in [64, 128, 256, 512]:
    print(f"bins={n}: {MI_continuous_continuous(transformed_event_log, 'seconds_in_day', 'duration_seconds', n, kde=cross_validation_kde)}")

In [ ]:
cvkde2 = lambda vals : cross_validation_kde(vals, n_sub=5000, cv=3)

for n in [64, 128, 256, 512]:
    print(f"bins={n}: {MI_continuous_continuous(transformed_event_log, 'seconds_in_day', 'duration_seconds', n, kde=cvkde2)}")

In [ ]:
cvkde2 = lambda vals : cross_validation_kde(vals, n_sub=10000, cv=2)

for n in [64, 128, 256, 512]:
    print(f"bins={n}: {MI_continuous_continuous(transformed_event_log, 'seconds_in_day', 'duration_seconds', n, kde=cvkde2)}")

In [ ]:
cvkde2 = lambda vals : cross_validation_kde(vals, n_sub=10000, cv=5)

for n in [64, 128, 256, 512]:
    print(f"bins={n}: {MI_continuous_continuous(transformed_event_log, 'seconds_in_day', 'duration_seconds', n, kde=cvkde2)}")

In [ ]:
cvkde2 = lambda vals : cross_validation_kde(vals, n_sub=10000, cv=5, n_runs=5, backend='multiprocessing')

for n in [64, 128, 256, 512]:
    print(f"bins={n}: {MI_continuous_continuous(transformed_event_log, 'seconds_in_day', 'duration_seconds', n, kde=cvkde2)}")

In [ ]:
cvkde2 = lambda vals : cross_validation_kde(vals, n_sub=15000, cv=5, n_runs=5, backend='multiprocessing')

for n in [64, 128, 256, 512]:
    print(f"bins={n}: {MI_continuous_continuous(transformed_event_log, 'seconds_in_day', 'duration_seconds', n, kde=cvkde2)}")

In [ ]:
cvkde2 = lambda vals : cross_validation_kde(vals, n_sub=15000, cv=5, n_runs=5, backend='multiprocessing')

for n in [2,4,8,16,32]:
    print(f"bins={n}: {MI_continuous_continuous(transformed_event_log, 'seconds_in_day', 'duration_seconds', n, kde=cvkde2)}")

In [ ]:
cvkde2 = lambda vals : cross_validation_kde(vals, n_sub=15000, cv=5, n_runs=5, backend='multiprocessing')

for n in [2048, 4096, 8192, 16384]:
    print(f"bins={n}: {MI_continuous_continuous(transformed_event_log, 'seconds_in_day', 'duration_seconds', n, kde=cvkde2)}")

In [ ]:
cvkde2 = lambda vals : cross_validation_kde(vals, n_sub=15000, cv=5, n_runs=5, backend='multiprocessing')

mi, p_x, p_y, p_xy = MI_continuous_continuous(transformed_event_log, 'seconds_in_day', 'duration_seconds', 2048, enable_tqdm=True, return_densities=True, kde=cvkde2)

In [ ]:
p_xy

In [ ]:
for n in [1024]:
    print(f"bins={n}: {MI_continuous_continuous(transformed_event_log, 'seconds_in_day', 'duration_seconds', n, kde=cross_validation_kde, enable_tqdm=True)}")

In [ ]:
for n in [4096, 256, 512]:
    print(f"bins={n}: {MI_continuous_continuous_CUDA(transformed_event_log, 'seconds_in_day', 'duration_seconds', n, enable_tqdm=True)}")

In [ ]:
for n in [1024]:
    print(f"bins={n}: {MI_continuous_continuous_CUDA(transformed_event_log, 'seconds_in_day', 'duration_seconds', n, kde=cross_validation_kde, enable_tqdm=True)}")

In [ ]:
for n in [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096, 4096*2, 4096*4]:
    print(f"n_neighbors={n}: {MI_discrete_continuous(transformed_event_log, 'concept:name', 'duration_seconds', n)}")

In [ ]:
for n in [4096, 4096*2, 4096*4]:
    print(f"bins={n}: {MI_discrete_continuous(transformed_event_log, 'concept:name', 'duration_seconds', n, kde=cross_validation_kde, enable_tqdm=True)}")

In [ ]:
for n in [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096, 4096*2, 4096*4]:
    print(f"bins={n}: {MI_discrete_continuous(transformed_event_log, 'org:resource_start', 'duration_seconds', n)}")

In [ ]:
for n in [4096, 4096*2, 4096*4]:
    print(f"bins={n}: {MI_discrete_continuous(transformed_event_log, 'org:resource_start', 'duration_seconds', n, kde=cross_validation_kde, enable_tqdm=True)}")

In [ ]:
for n in [1024, 2048, 4096, 4096*2, 4096*4]:
    print(f"bins={n}: {MI_discrete_continuous(transformed_event_log, 'day_of_week', 'duration_seconds', n, kde=cross_validation_kde, enable_tqdm=True)}")